# Testing Masteller (2025) equation to La Jara data 

In [5]:
# import libraries
import numpy as np
import pandas as pd

Model Function

In [ ]:
def evolve_tau_c_continuous(
    tau_star,           # array of Shields stress τ*(t)
    tau_c0,             # initial τ*c
    dt,                 # timestep (same units as k1, k2)
    k1, k2, epsilon,
    gamma=2.5,
    tau_c_min=0.0343,
    tau_c_max=0.3435
):
    """
    forward-model τ*c evolution using Masteller et al. (2025)
    will return a predicted τ*c ndarray
    """
    tau_c = np.zeros_like(tau_star) # initialize output array
    tau_c[0] = tau_c0 # set initial tau*c

    for i in range(1, len(tau_star)):
        tc = tau_c[i-1] # previous tau*c
        t = tau_star[i] # current tau*
        # enforce bounds of min and max critical shear stress
        tc = np.clip(tc, tau_c_min, tau_c_max)

        # calculate feedback parameter B
        B = (tau_c_max - tc) / (tau_c_max - tau_c_min)
        # transport capacity
        R = t / tc
        # strengthening term
        strengthen = k1 * B / (1.0 + R**(-gamma))
        # weakening term
        if R > 1.0:
            weaken = k2 * (1 - B) * (R - 1)**epsilon
        else:
            weaken = 0.0

        # forward euler update
        d_tau_c = (strengthen - weaken) * dt
        tau_c[i] = tc + d_tau_c

    return tau_c

In [ ]:
def evolve_tau_c_gap_reset(
    tau_star,           # array of Shields stress τ*(t); np.nan where τ* is missing
    event_indices,      # integer indices where τ*c observations are available
    tau_c_obs,          # observed τ*c values at event_indices
    dt,                 # timestep between τ* samples (15 minutes in my case)
    k1, k2, epsilon,    # model parameters to calibrate
    gamma=2.5,          # fixed exponent from literature 
    tau_c_min=0.0343,   # minimum τ*c
    tau_c_max=0.3435    # maximum τ*c
):
    # initialize modeled τ*c array
    tau_c = np.full_like(tau_star, np.nan, dtype=float) # nan means τ*c is undefined (e.g., during gaps)
    tau_c_obs_dict = dict(zip(event_indices, tau_c_obs)) # dictionary of observed τ*c values
    has_tau = ~np.isnan(tau_star) # boolean array indicating where τ* is available

    i = 0
    while i < len(tau_star):

        # case 1: no τ* → do nothing
        if not has_tau[i]: # τ* missing: τ*c evolution is undefined
            i += 1         # leave tau_c[i] as NaN
            continue

        # case 2: τ* resumes after gap → reinitialize τ*c if available
        if (has_tau[i] and (i == 0 or not has_tau[i - 1]) and i in tau_c_obs_dict):
            # τ* just resumed AND we have an observed τ*c
            tau_c_current = tau_c_obs_dict[i]
            tau_c[i] = tau_c_current
            i += 1
            continue

        # case 3: normal forward evolution
        if i > 0 and not np.isnan(tau_c[i - 1]):
            tau_c_current = tau_c[i - 1] # previous τ*c
        else:
            # τ*c is not yet defined (e.g., τ* resumed but no τ*c observed yet)
            i += 1
            continue

        t = tau_star[i] # current τ*
        tc = np.clip(tau_c_current, tau_c_min, tau_c_max) # enforce bounds on τ*c
        
        # calculate feedback parameter B
        B = (tau_c_max - tc) / (tau_c_max - tau_c_min)
        # transport capacity
        R = t / tc
        # strengthening term
        strengthen = k1 * B / (1.0 + R**(-gamma))
        # weakening term
        if R > 1.0:
            weaken = k2 * (1 - B) * (R - 1)**epsilon
        else:
            weaken = 0.0

        # forward euler update
        d_tau_c = (strengthen - weaken) * dt
        tau_c[i] = tc + d_tau_c
        i += 1

    return tau_c


Parameter Calibration [(grid search)](https://en.wikipedia.org/wiki/Hyperparameter_optimization)


In [ ]:
def calibrate_parameters(
    tau_star,
    event_indices,      # indices of event start times
    tau_c_obs,          # observed τ*c at events
    dt,
    k1_vals,
    k2_vals,
    eps_vals,
    gamma,
    tau_c_min,
    tau_c_max
):
    """
    grid-search calibration of the model using mean absolute error (MAE) 
    """
    results = []
    for k1 in k1_vals:
        for k2 in k2_vals:
            for eps in eps_vals:

                tau_c_model = evolve_tau_c_gap_reset(tau_star, event_indices, tau_c_obs, dt, k1, k2, eps, gamma, tau_c_min, tau_c_max)
                # extract modeled τ*c at event starts
                tau_c_pred = tau_c_model[event_indices]
                # calculate mean absolute error (paper uses MAE), and ignore NaNs
                mae = np.nanmean(np.abs(tau_c_pred - tau_c_obs))
                results.append({
                    "k1": k1,
                    "k2": k2,
                    "epsilon": eps,
                    "MAE": mae
                })
    return results


Parameter Ranges 

In [8]:
# start, stop, number of values
k1_vals = np.logspace(-6, -2, 40) # values evenly spaced on log scale
k2_vals = np.logspace(-6, -2, 40)
eps_vals = np.arange(1, 11) # integer values from 1 to 10

Import Data

In [ ]:
# import data 
data = pd.read_csv('lajara_antecedent.csv')

In [ ]:
# store everything with datetimes
df_tau = pd.DataFrame(
    {"tau_star": tau_star_values},
    index=pd.DatetimeIndex(tau_times)
)

df_events = pd.DataFrame(
    {"tau_c_obs": tau_c_obs},
    index=pd.DatetimeIndex(event_times)
)

# convert times to indices
df_tau = df_tau.sort_index() # ensure sorted and aligned

# build integer index mapping
time_to_index = pd.Series(
    np.arange(len(df_tau)), 
    index=df_tau.index
)

event_indices = time_to_index.loc[df_events.index].to_numpy() # integer indices of event times
tau_c_obs = df_events["tau_c_obs"].to_numpy() # observed τ*c values
tau_star = df_tau["tau_star"].to_numpy() # τ* time series array


### Selecting best fit parameters

In [ ]:
results = calibrate_parameters(
    tau_star,
    event_indices,
    tau_c_obs,
    tau_c0=tau_c_obs[0],
    dt=10.0,               # minutes, for example
    k1_vals=k1_vals,
    k2_vals=k2_vals,
    eps_vals=eps_vals,
    gamma=2.5,
    tau_c_min=0.03,
    tau_c_max=0.36
)

best = min(results, key=lambda x: x["MAE"])

print("Best-fit parameters:")
print(best)
